In [ ]:
import numpy as np
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score
from pathlib import Path
import pandas as pd

In [ ]:
data_path = Path("/Users/patrykksiazek/Desktop/Machine-Learning---Preliminary-Data-Analysis/")

In [ ]:
grouped_train_logs = pd.read_csv(data_path / 'grouped_train_logs_advanced.csv').drop(columns={'Unnamed: 0'})
X_train_scores = pd.read_csv(data_path / 'X_train_scores.csv').drop(columns={'Unnamed: 0'})
X_test_scores = pd.read_csv(data_path / 'X_test_scores.csv').drop(columns={'Unnamed: 0'})

In [ ]:
cols_to_drop = ['id', 'max_up_time', 'min_down_time', 'activity_len_per_essay_Nonproduction']

In [ ]:
# Splitting the grouped_train_logs into train and test sets based on IDs
train_data = grouped_train_logs.merge(X_train_scores, on='id').drop(columns=cols_to_drop)
test_data = grouped_train_logs.merge(X_test_scores, on='id').drop(columns=cols_to_drop)

In [ ]:
# Separating features and target variable
X_train = train_data.drop(['score'], axis=1)
y_train = train_data['score']
X_test = test_data.drop(['score'], axis=1)
y_test = test_data['score']

## XGBoost

In [ ]:
import xgboost as xgb

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=0.1, random_state=42)

In [ ]:
dtrain = xgb.DMatrix(X_train, label=y_train)
dvalid = xgb.DMatrix(X_valid, label=y_valid) 
dtest = xgb.DMatrix(X_test, label=y_test) 

In [ ]:
param = {'max_depth': 2}
evallist = [(dtrain, 'train'), (dvalid, 'eval')]

In [ ]:
num_round = 20
bst = xgb.train(param, dtrain, num_round, evallist)

In [ ]:
xgb.plot_importance(bst)
plt.show()

In [ ]:
y_test_pred = bst.predict(dtest)

In [ ]:
np.sqrt(mean_squared_error(y_test, y_test_pred))

## Pewność modelu

In [ ]:
y_train_pred = bst.predict(dtrain)

In [ ]:
train_rests = y_train_pred - y_train

In [ ]:
train_rests

In [ ]:
train_res = X_train.merge(train_rests, left_index=True, right_index=True)

In [ ]:
train_res

In [ ]:
train_res = train_res.merge(y_train, left_index=True, right_index=True)

In [ ]:
train_res['y_train_pred'] = y_train_pred
train_res.rename(columns={'score_x': 'rest', 'score_y': 'y_train'}, inplace=True)

In [ ]:
train_res

In [ ]:
train_res.corr()['rest'].sort_values()

In [ ]:
train_res['writing_time_no_pause'].hist()

In [ ]:
for t in [30000, 50000, 70000, 100000]:
    subs = train_res[train_res['writing_time_no_pause'] < t]
    print(np.sqrt(mean_squared_error(subs['y_train'], subs['y_train_pred'])), len(subs) / len(train_res))

In [ ]:
test_res = X_test.merge(y_test, left_index=True, right_index=True)
test_res['y_test_pred'] = y_test_pred

In [ ]:
test_res

In [ ]:
subs = test_res[test_res['writing_time_no_pause'] < 50000]
print(np.sqrt(mean_squared_error(subs['score'], subs['y_test_pred'])), len(subs) / len(test_res))

In [ ]:
subs['score'].hist(bins=14)

## Wyjaśnialność 

In [ ]:
import shap

In [ ]:
explainer = shap.Explainer(bst)

In [ ]:
shap_values = explainer.shap_values(X_train)

In [ ]:
shap_values

In [ ]:
shap.summary_plot(shap_values, X_train, feature_names=X_train.columns)